# TimesNet & TimeMixer — M4 Weekly Forecast Demo (tfts pipeline)

End-to-end demo of the `tfts` TensorFlow ports of **TimesNet** and **TimeMixer** on the M4 **Weekly** short-term forecasting task (`seq_len=26`, `pred_len=13`), using the exact training recipe the repo benchmark (`exps/ts_m4_parity/`) measured to beat the PyTorch reference (`reference/Time-Series-Library`).

Everything model/training related comes from the **tfts library itself**:

- model architecture + defaults from each `tfts` `Config` class,
- training via `tfts.training.WindowedTrainer` — elementwise SMAPE loss, cosine-annealed Adam over the full epoch budget, best-validation snapshot + early stopping, fresh random windows per epoch, final-window held-out test.

The TimesNet/TimeMixer ports run raw `tf` ops at the top level of `__call__`, so `WindowedTrainer` pre-builds the model with one real forward and wraps it in a real-tensor `tf.keras.Model` (they are not `build_model(Input)`-symbolic safe).

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../.."))                  # repo root -> ./tfts
sys.path.insert(0, os.path.abspath("../../exps/phase2_tfts"))  # -> M4 loader

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tfts import WindowedTrainer, final_windows
from tfts.models.timesnet import TimesNet, TimesNetConfig
from tfts.models.timemixer import TimeMixer, TimeMixerConfig

from m4_pipeline import load_weekly   # M4 data loader (data prep only)

tf.keras.utils.set_random_seed(2026)
tf.random.set_seed(2026)

SEQ_LEN, PRED_LEN = 26, 13


## 1. Data — M4 Weekly
359 weekly series; each series history is the training data and the held-out test target is the official 13-step horizon.

In [ ]:
ids, histories, targets = load_weekly()
print("series:", len(ids), "| max history len:", max(len(h) for h in histories))
print("test target shape (batch, horizon):", targets.shape)
print("history example[-5:]:", histories[0][-5:].round(2))


## 2. Training pipeline — `tfts.WindowedTrainer`
Training settings per model only (architecture comes from the tfts `Config` defaults, already tuned to this recipe). `WindowedTrainer` encapsulates the whole recipe: fresh windows per epoch, elementwise SMAPE + mask, cosine LR, best-val snapshot + early stop.

In [ ]:
# Training settings only; default model architecture = tfts Config defaults.
TRAIN = {
    "timesnet": dict(batch_size=16, lr=0.001, epochs=40, patience=10),
    "timemixer": dict(batch_size=128, lr=0.01, epochs=100, patience=30),
}


def build_model(name):
    if name == "timesnet":
        return TimesNet(predict_sequence_length=PRED_LEN, config=TimesNetConfig())
    return TimeMixer(predict_sequence_length=PRED_LEN, config=TimeMixerConfig())


def train_eval(name, seed=2026):
    tf.keras.utils.set_random_seed(seed)
    tf.random.set_seed(seed)
    w = TRAIN[name]
    wt = WindowedTrainer(build_model(name), seq_len=SEQ_LEN, pred_len=PRED_LEN,
                         lr=w["lr"], batch_size=w["batch_size"],
                         epochs=w["epochs"], patience=w["patience"],
                         seed=seed, lr_schedule="cosine")
    logs = wt.train(histories, targets)
    ev = wt.evaluate(histories, targets)
    return dict(name=name, smape=float(ev["smape"]), mae=float(ev["mae"]),
                mse=float(ev["mse"]), preds=ev["prediction"], logs=logs)


## 3. TimesNet (default recipe)

In [ ]:
res_tn = train_eval("timesnet")
print(f"{res_tn['name']:<10s} SMAPE={res_tn['smape']:.3f}  MAE={res_tn['mae']:.1f}  MSE={res_tn['mse']:.1f}")


## 4. TimeMixer (default recipe)

In [ ]:
res_tm = train_eval("timemixer")
print(f"{res_tm['name']:<10s} SMAPE={res_tm['smape']:.3f}  MAE={res_tm['mae']:.1f}  MSE={res_tm['mse']:.1f}")


## 5. Training curves (validation SMAPE vs epoch)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for r, c in [(res_tn, "tab:blue"), (res_tm, "tab:green")]:
    e = [l["epoch"] for l in r["logs"]]
    v = [l["val"] for l in r["logs"]]
    ax.plot(e, v, marker="o", color=c, label=f'{r["name"]} val SMAPE')
ax.set_xlabel("epoch")
ax.set_ylabel("SMAPE")
ax.legend()
ax.set_title("M4 Weekly validation SMAPE (test: "
              + ", ".join(f'{r["name"]}={r["smape"]:.2f}' for r in (res_tn, res_tm)) + ")")
plt.tight_layout()
plt.show()


## 6. Multi-step forecasts on the held-out test window

In [ ]:
def plot_forecast(res, s=0):
    x_test, _ = final_windows(histories, SEQ_LEN)
    history = x_test[s, :, 0]
    fut = np.arange(SEQ_LEN, SEQ_LEN + PRED_LEN)
    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(-SEQ_LEN, 0), history, label="history")
    plt.plot(fut, targets[s], label="actual", marker="o")
    plt.plot(fut, res["preds"][s], label="forecast", marker="x")
    plt.axvline(0, color="gray", ls="--")
    plt.legend()
    plt.xlabel("time (relative to forecast start)")
    plt.ylabel("value")
    plt.title(f'{res["name"].title()} series {s}: {PRED_LEN}-step forecast (SMAPE {res["smape"]:.2f})')
    plt.tight_layout()
    plt.show()


plot_forecast(res_tn)
plot_forecast(res_tm)
